[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Automatic Differentiation &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


**1.** Find the derivative of y = 5x² − 3x at x = 2 with PyTorch and with TensorFlow, and check it against 10x − 3.

In [2]:
x = torch.tensor(2.0, requires_grad=True)
(5 * x ** 2 - 3 * x).backward()

x_tf = tf.Variable(2.0)
with tf.GradientTape() as tape:
    y_tf = 5 * x_tf ** 2 - 3 * x_tf

print(x.grad.item(), float(tape.gradient(y_tf, x_tf)), 10 * 2.0 - 3)

17.0 17.0 17.0


Both frameworks give 17, which is 10 × 2 − 3.

**2.** Make `x = torch.linspace(-2, 2, 5, requires_grad=True)`, find the gradient of `(x ** 3).sum()`, and check it against 3x² with `torch.allclose`.

In [3]:
x = torch.linspace(-2, 2, 5, requires_grad=True)
(x ** 3).sum().backward()
print(x.grad)
print(torch.allclose(x.grad, 3 * x.detach() ** 2))

tensor([12.,  3.,  0.,  3., 12.])
True


The gradient is 3x² at each of the five points, 12, 3, 0, 3 and 12, and `torch.allclose` confirms it.

**3.** Print the operations behind `torch.tensor(2.0, requires_grad=True).exp() * 3` with `steps_back`, defined again in your cell, and find its gradient, which should equal 3e².

In [4]:
import math


def steps_back(tensor):
    """The names of the operations behind a tensor, following the first input of each."""
    names, step = [], tensor.grad_fn
    while step is not None:
        names.append(step.name())
        inputs = [fn for fn, _ in step.next_functions if fn is not None]
        step = inputs[0] if inputs else None
    return names


start = torch.tensor(2.0, requires_grad=True)
result = start.exp() * 3
print(steps_back(result))
result.backward()
print(round(start.grad.item(), 4), round(3 * math.exp(2), 4))

['MulBackward0', 'ExpBackward0', 'torch::autograd::AccumulateGrad']
22.1672 22.1672


The record reads back from the multiplication by 3 to the exponential to the leaf, and the gradient, 22.1672, equals 3e² to the four places printed.

**4.** With one persistent tape, find the gradients of `v ** 2` and of `tf.sin(v)` at v = 0.5, and check them against 2v and cos v.

In [5]:
v = tf.Variable(0.5)
with tf.GradientTape(persistent=True) as tape:
    square = v ** 2
    wave = tf.sin(v)
print(round(float(tape.gradient(square, v)), 4), 2 * 0.5)
print(round(float(tape.gradient(wave, v)), 4), round(float(tf.cos(0.5)), 4))
del tape

1.0 1.0
0.8776 0.8776


The tape gave 1.0 for `v ** 2`, which is 2v at 0.5, and 0.8776 for `tf.sin(v)`, which is cos 0.5.

**5.** Compute the mean squared error of the line with a slope of −2 and an intercept of −5 on the test stations inside `torch.no_grad()`, and print the error and its `requires_grad`.

In [6]:
w = torch.tensor(-2.0, requires_grad=True)
b = torch.tensor(-5.0, requires_grad=True)
with torch.no_grad():
    predicted = w * torch.from_numpy(scaled[test_rows, 0]) + b
    test_error = ((predicted - torch.from_numpy(jan_mean[test_rows])) ** 2).mean()
print(round(test_error.item(), 3), test_error.requires_grad)

4.554 False


The line's mean squared error on the test stations is 4.554, and the result does not require a gradient, because it was computed inside `torch.no_grad()`.

---

&#8592; **Back to:** [Automatic Differentiation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/04-automatic-differentiation.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
